# 3. Incidents, Investigation, and Automated Response

### What you'll learn
- How alerts correlate into incidents (the SOC unit of work) — and where naive
  correlation falls apart
- The four-step investigation workflow (triage -> investigate -> respond -> close)
- The bad -> best progression from manual response to automated playbooks
- How threat-intelligence watchlists add context to investigations

## From alerts to incidents

One real attack produces **many alerts** across products. The goal is to land them in
ONE incident:

```
Alert: Brute force sign-in        entity UserPrincipalName=alice@contoso.com  -+
Alert: Sign-in from Moscow        entity (none)                                +-> Incident: alice compromised
Alert: Lateral movement           entity AccountName=alice                    -+
```

**Our engine does not achieve that, and you should see exactly why.** `/incidents/correlate`
groups alerts whose entity dictionary is *byte-identical*. Those three alerts describe the
same human being, but they carry `{"UserPrincipalName": "alice@contoso.com"}`,
`{}`, and `{"AccountName": "alice"}` — three different keys — so they become three
separate incidents.

That is not a toy problem. It is the single hardest part of real correlation:
**entity normalisation**. `alice`, `alice@contoso.com`, `CONTOSO\alice` and an Entra
object GUID are one identity, and something has to know that before grouping can work.
Sentinel does it with an entity-mapping schema per rule; Defender XDR does it with a
correlation engine that resolves accounts, devices and IPs across products.

The cell below measures the fragmentation instead of asserting the happy path. Lab 2
(`02-incident-response`) then works the problem: it links incidents by *shared* entities
rather than identical ones.

## 0. Setup — pick the lab kernel

This lab has its own `uv`-managed virtual environment. Before running any code cell:

1. From `security-certs/sc-200/01-build-a-siem/` run once in a terminal:
   ```bash
   uv sync
   docker compose up -d
   ```
2. In VS Code, click the kernel picker (top-right of this notebook) and choose **`.venv (Python 3.xx)`** from this folder.
3. If the kernel does not appear, reload the window: `Cmd+Shift+P` → `Reload Window`.

The log-generator container has already seeded the SIEM with normal traffic **and** four attack patterns (brute force, lateral movement, exfiltration, phishing). Every cell below talks to `http://localhost:8000`.

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

def _j(v):
    return json.loads(v) if isinstance(v, str) else (v or {})

incidents = httpx.get(f'{SIEM}/incidents').json()
assert incidents, 'no incidents - run `docker compose up -d` and let sc200-log-generator finish'

print(f'=== Open Incidents ({len(incidents)}) ===\n')
for inc in incidents:
    sev = {'Critical':'🟣','High':'🔴','Medium':'🟡','Low':'🟢'}.get(inc['severity'], '⬜')
    alerts = _j(inc['alert_ids'])
    entities = _j(inc['entities'])
    print(f'{sev} {inc["id"]} [{inc["status"]}]  {inc["title"]}')
    print(f'   Alerts: {len(alerts)}  |  Entities: {entities}  |  Assigned: {inc["assigned_to"] or "Unassigned"}\n')


In [ ]:
# --- Measure the fragmentation the markdown above describes ---------------------
# Find every incident whose alerts mention alice, however she happens to be spelled.
ALICE_SPELLINGS = {'alice', 'alice@contoso.com'}

def mentions_alice(incident_id):
    detail = httpx.get(f'{SIEM}/incidents/{incident_id}').json()
    for a in detail['alerts']:
        for ev in _j(a['evidence']) or []:
            for key in ('UserPrincipalName', 'AccountName'):
                if ev.get(key) in ALICE_SPELLINGS:
                    return True
    return False

alice_incidents = [i for i in incidents if mentions_alice(i['id'])]
print(f'Incidents whose evidence names alice: {len(alice_incidents)}')
for i in alice_incidents:
    print(f'   {i["id"]}  entities={_j(i["entities"]) or "{}"}  {i["title"]}')

# One attack, N tickets. If this ever collapses to 1, the correlation engine got
# smarter and the lesson below needs rewriting - so fail loudly rather than quietly
# teaching something that is no longer true.
assert len(alice_incidents) > 1, (
    'expected one campaign to fragment across several incidents; entity-identical '
    'grouping apparently merged them - re-read /incidents/correlate'
)
print(f'\n→ ONE attack, {len(alice_incidents)} incidents. The evidence rows all name alice;')
print('  the incident ENTITIES do not agree on how to spell her, so nothing grouped.')
print('  Fix in a real SIEM: entity mapping normalises the account before correlation.')


## 3.1 Investigation workflow

1. **Triage** — severity OK? real? assign an analyst.
2. **Investigate** — drill into alerts, entities, timeline.
3. **Contain & Remediate** — disable users, isolate devices, block IPs.
4. **Close** — classify (TruePositive / BenignPositive / FalsePositive / Undetermined) and document.

In [ ]:
# Pick the brute-force incident BY NAME, not by position.
#   `incidents` comes back ordered by created_at, and every alert in the seed run is
#   created within the same second - so "the first High incident" is whichever row
#   SQLite happened to return. The narrative in the cells below (brute force, disable
#   the account) is only true for one specific incident, so select that one explicitly.
target = next((i for i in incidents if 'Brute force sign-in' in i['title']), None)
assert target is not None, (
    'no brute-force incident found. Titles present: '
    + str([i['title'] for i in incidents])
)
target_entities = _j(target['entities'])
assert 'UserPrincipalName' in target_entities, (
    f'expected a user entity to pivot on, got {target_entities}'
)

details = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
print(f'=== Investigating: {details["id"]} ===')
print(f'Title: {details["title"]}')
print(f'Severity: {details["severity"]}  |  Status: {details["status"]}')
print(f'\n--- Related alerts ({len(details["alerts"])}) ---')
for alert in details['alerts']:
    ev = _j(alert['evidence']) or []
    print(f'  🚨 {alert["title"]}')
    print(f'     Tactic: {alert["tactic"]}  |  Severity: {alert["severity"]}')
    for row in ev:
        # An alert carries a SAMPLE of the matching rows, not all of them.
        print(f'       · {row.get("IPAddress", "-"):<16} {row.get("Location", "-"):<12} '
              f'{row.get("ResultType", "-"):<8} risk={row.get("RiskLevel", "-")}')
    print()

print('Read the evidence, not the headline count. The alert stores a SAMPLE of the')
print('matching rows, and the sample is what turns a number into a finding: every')
print('row shares one external source address, one location and a non-zero risk')
print('level. "15 failures" alone could be a broken password manager. "15 failures')
print('from one address in Moscow" is a targeted attack, and only the evidence says so.')
print('Note also what the rule did NOT check: it groups by user and ignores the source')
print('IP entirely, so a user with scattered typos would be counted the same way.')
print('Grouping by (user, IP) would be the next tuning iteration.')


In [ ]:
# Step 1 - Triage: assign, activate, comment
httpx.patch(f'{SIEM}/incidents/{target["id"]}', json={
    'status': 'Active',
    'assigned_to': 'soc-analyst-1@contoso.com',
    'comment': 'Triaged - looks like a real brute force, investigating.',
})
updated = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
print(f'Status: {updated["status"]}  |  Assigned: {updated["assigned_to"]}')
comments = _j(updated['comments']) or []
print('Comments:', comments)

assert updated['status'] == 'Active' and updated['assigned_to'], 'triage did not persist'


In [ ]:
# Step 2 - Investigate: pivot from entity to all related activity
user = target_entities['UserPrincipalName']

print(f'--- Sign-in breakdown for {user} ---')
r = httpx.post(f'{SIEM}/query', json={'table_name':'SigninLogs','filter':{'UserPrincipalName':user},'aggregate_by':'ResultType'})
signin_counts = {row['group_key']: row['count'] for row in r.json()['results']}
for k, v in signin_counts.items():
    print(f'  {k}: {v}')

print(f'\n--- Endpoint activity for {user.split("@")[0]} ---')
r = httpx.post(f'{SIEM}/query', json={'table_name':'DeviceEvents','filter':{'AccountName':user.split("@")[0]},'limit':200})
endpoint = r.json()['results']
ATTACK_TOOLS = {'mimikatz.exe', 'psexec.exe'}

# Show the known-bad tooling first. Sorting by timestamp buries it: the most RECENT
# rows are the bulk exfiltration uploads, so a plain "last 10" hides the mimikatz runs
# entirely - a small display choice that quietly loses the finding.
flagged = [e for e in endpoint if e['FileName'] in ATTACK_TOOLS]
for log in flagged:
    print(f'  ⚠️ {log["DeviceName"]:<14} {log["FileName"]:<14} ({log["ActionType"]})')
print('  --- everything else, by process ---')
counts = {}
for e in endpoint:
    if e['FileName'] not in ATTACK_TOOLS:
        counts[e['FileName']] = counts.get(e['FileName'], 0) + 1
for name, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f'     {name:<14} {n} event(s)')

# The pivot is the whole point of entity mapping: the sign-in alert should lead you
# to endpoint activity you were not alerted about. Assert it actually does.
tools_seen = {e['FileName'] for e in flagged}
hosts = {e['DeviceName'] for e in flagged}
assert signin_counts.get('Failure', 0) >= 5, f'expected the brute-force failures, saw {signin_counts}'
assert signin_counts.get('Success', 0) >= 1, 'the brute force should have ended in a success'
assert tools_seen, 'pivoting from the user found no attacker tooling - the pivot taught nothing'
print(f'\n→ The sign-in alert led to {sorted(tools_seen)} across {len(hosts)} host(s): {sorted(hosts)}')
print('  Nothing paged you about those. That is why you pivot on the entity instead')
print('  of closing the alert you were handed.')


## 3.2 Bad response -> Best response (automation)

Manual response is slow. At 3am, it also does not happen. Mature SOCs encode the runbook into a **playbook** that runs automatically.

| Approach | Mean-time-to-respond | Reliability |
|---------|---------------------|-------------|
| 🔴 Manual (analyst logs in, clicks through portals) | Minutes to hours | Low — depends on staffing |
| 🟡 Scripted one-off (Python script per alert type) | Minutes | Medium — scripts drift |
| ✅ **Playbook** (Logic App triggered by the SIEM) | Seconds | High — audited, version-controlled |

In [ ]:
# BAD: print what an analyst WOULD do manually for this incident.
print('Manual runbook (what an analyst would do by hand):')
print('  1. Open Entra admin portal')
print('  2. Find the user')
print('  3. Disable the account')
print('  4. Revoke sessions')
print('  5. Email the SOC channel')
print('Typical time: 5-15 minutes, IF the analyst is awake.')

In [ ]:
# BEST: trigger a playbook. One API call, executes every step instantly.
r = httpx.post(f'{SIEM}/playbooks/run/{target["id"]}').json()
for pb in r['playbooks_executed']:
    print(f'🤖 Playbook: {pb["playbook"]}')
    for i, action in enumerate(pb['actions'], 1):
        print(f'   Step {i}: ✅ {action["description"]}')

# A playbook that silently matches nothing is the automation equivalent of a rule that
# never fires - it looks fine right up until the night you needed it.
assert r['playbooks_executed'], (
    f"no playbook matched severity={target['severity']} + this incident's tactics. "
    'Check /playbooks trigger_severity / trigger_tactic.'
)
print('\n💡 In real Sentinel, playbooks are Logic Apps that call Azure APIs:')
print('   - Disable user   -> Microsoft Graph API')
print('   - Isolate device -> Defender for Endpoint API')
print('   - Block IP       -> Azure Firewall API')
print('   - Notify SOC     -> Teams / ServiceNow webhook')


## 3.3 Threat-intelligence watchlists (IOC matching)

A **watchlist** is a lookup table of known-bad (or known-good) values you import into the SIEM. Typical uses:

- 🕷️ Known-bad IPs from a threat feed (e.g., AlienVault OTX, Microsoft TI)
- 👑 VIP users who need extra monitoring
- 🧑‍💻 Terminated employees (any activity = urgent)
- 🗝️ Approved admin tools (allowlist)

In KQL the pattern is `Table | where Column in (watchlist)`. Let's do it in our mini-SIEM.

In [ ]:
# Create a TI watchlist of known-bad IPs
httpx.post(f'{SIEM}/watchlists', json={
    'name': 'known_bad_ips',
    'description': 'Simulated threat-intel feed (Tor exits, C2 servers)',
    'items': ['185.220.101.42', '45.33.32.156', '198.51.100.99'],
})

# Match the watchlist against sign-in logs
r = httpx.post(f'{SIEM}/watchlists/match', json={
    'watchlist': 'known_bad_ips',
    'table_name': 'SigninLogs',
    'field': 'IPAddress',
    'time_range_minutes': 1440,
}).json()
print(f'🔍 {r["match_count"]} sign-ins from known-bad IPs:')
for m in r['matches'][:5]:
    print(f'  {m["UserPrincipalName"]:<20} {m["IPAddress"]:<16} {m["ResultType"]:<8} {m["Location"]}')

assert r['match_count'] > 0, 'TI watchlist matched nothing - the IOC list and the seed data disagree'
print('\n💡 Real Sentinel ships a free Microsoft TI feed plus the ThreatIntelligenceIndicator table.')


In [ ]:
# Step 4 - Close the incident with a classification
httpx.patch(f'{SIEM}/incidents/{target["id"]}', json={
    'status': 'Closed',
    'classification': 'TruePositive',
    'comment': 'Confirmed brute force. Account disabled via playbook. Password reset required.',
})
final = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
print(f'Status: {final["status"]}  |  Classification: {final["classification"]}')
for c in (_j(final['comments']) or []):
    print(f'  - {c["text"]}')

assert final['status'] == 'Closed' and final['classification'] == 'TruePositive', \
    'a closed incident must carry BOTH a status and a classification'

print('\n--- Incident classifications ---')
print('  TruePositive   - confirmed attack, action taken')
print('  BenignPositive - real activity, not malicious (e.g., pen test)')
print('  FalsePositive  - detection was wrong -> tune the rule')
print('  Undetermined   - not enough evidence')
print('\n⚠️  Note what we did NOT close: the lateral-movement, credential-dumping and')
print('    exfiltration incidents from the same campaign are still open, because this')
print('    SIEM never grouped them with the brute force. Closing "the" incident on a')
print('    fragmented campaign is how attackers keep their foothold. Lab 2 fixes it.')


## Automation rules vs playbooks (exam)

| | Automation rules | Playbooks |
|-|-----------------|----------|
| **What** | Lightweight if/then logic | Full Logic App workflows |
| **Trigger** | Incident create/update, alert create | Called by automation rules or manually |
| **Actions** | Change severity, assign, run playbook, close | Any Azure/external API call |
| **Code** | No-code (portal UI) | Low-code (Logic Apps designer) |
| **Use case** | Triage automation, suppress noise | Complex response workflows |

- **Automation rules** decide *when* to run a playbook.
- **Playbooks** define *what actions* to take.
- Automation rules can also suppress, re-assign, and close incidents without playbooks.
- Limit: **512 automation rules** per workspace.

---

## What you built

- ✅ Data ingestion from 4+ sources (plus your own custom connector)
- ✅ Query engine with filter, aggregation, time window (KQL-like)
- ✅ Analytics rules, bad -> best progression, false-positive tuning
- ✅ MITRE ATT&CK coverage mapping
- ✅ Alert -> Incident correlation
- ✅ Full investigation workflow with automated playbooks
- ✅ Threat-intelligence watchlists / IOC matching

## What the mini-SIEM deliberately does NOT do

Sentinel does all of the above at scale with KQL, Logic Apps and ML. Know where the toy
stops, because every gap below is a real engineering problem:

| Mini-SIEM | Real Sentinel |
|-----------|---------------|
| Correlates alerts whose entity dict is **byte-identical** | Maps entities per rule, then groups by *shared* entity — `alice` and `alice@contoso.com` resolve to one account |
| Filters are exact-value equality on one JSON field | Full KQL: joins, `in~`, regex, time-series, `arg_max`, ML functions |
| Rules run only when you `POST /rules/evaluate` | Scheduled every 5 min with a lookback window, and near-real-time rules |
| Every evaluation re-fires every rule (alerts pile up) | Alert grouping, event grouping and suppression windows |
| Playbooks "execute" by writing a row | Logic Apps that really do call Graph / Defender / firewall APIs |
| No cost model | Per-GB ingestion billing, Basic/Auxiliary tiers, DCR transformations |

The single biggest gap is the first row, and notebook 3 measured it: one campaign against
one user produced four unrelated incidents. Lab 2 works that problem.

**Next lab**: [02 — Incident Response](../../02-incident-response/)